## LLM Priors Assessments

### Define functions to generate descriptions and priors for synthetic datasets

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'priors').exists() else cwd.parent
if not (REPO_ROOT / 'priors').exists():
    raise RuntimeError(f'Could not locate repository root from {cwd}')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'Repository root: {REPO_ROOT}')


In [ ]:
import re
from pathlib import Path

import pandas as pd
import pyagrum as gum
from pydantic import BaseModel, Field
from typing import Annotated
from tqdm.asyncio import tqdm

from priors.llm import extract
from priors.prompt import prepare_priors


def extract_facts(string: str) -> list[dict]:
    pattern = re.compile(r"ext_((in)?dep)\((.+)\). I=([01].\d+), NA\n")
    matches = pattern.findall(string)
    facts = []
    for match in matches:
        cit_type, _, triple, score = match
        X, Y, S = triple.split(",")
        facts.append(
            {
                "cit_type": cit_type,
                "X": int(X),
                "Y": int(Y),
                "S": set() if S == "empty" else {int(var) for var in S[1:].split("y")},
                "score": float(score),
            }
        )
    return pd.DataFrame(facts).sort_values(
        by="score", ascending=False, ignore_index=True
    )


async def generate_priors(
    bn: gum.BayesNet,
    variable_descriptions: dict[str, str] | None = None,
    prior_model: str = "gemini-2.5-flash",
    parse_model: str = "gemini-2.5-flash-lite",
) -> dict:
    seed = 2025
    gum.initRandom(seed=seed)

    priors_prompt = prepare_priors(bn, descriptions=variable_descriptions)
    priors_raw = None
    while priors_raw is None:
        priors_raw = (
            (await extract(priors_prompt, None, model=prior_model))
            .choices[0]
            .message.content
        )

    valid_var_pattern = r"|".join(re.escape(var) for var in bn.names())
    VarType = Annotated[str, Field(pattern=valid_var_pattern)]

    class VarConstraints(BaseModel):
        forbidden: set[tuple[VarType, VarType]] = set()
        required: set[tuple[VarType, VarType]] = set()

    priors = await extract(
        prompt=priors_raw,
        model=parse_model,
        pydantic_model=VarConstraints,
    )
    true_arrows = {
        (bn.variable(id1).name(), bn.variable(id2).name()) for id1, id2 in bn.arcs()
    }

    # ----- Forbidden metrics -----
    forbidden_length = len(priors.forbidden)
    forbidden_precision = len(priors.forbidden - true_arrows) / max(forbidden_length, 1)
    forbidden_recall = len(priors.forbidden - true_arrows) / (
        bn.size() * (bn.size() - 1) - len(true_arrows)
    )
    forbidden_f1 = (
        2
        * forbidden_precision
        * forbidden_recall
        / max(forbidden_precision + forbidden_recall, 1e-6)
    )

    # ----- Required metrics -----
    required_length = len(priors.required)
    required_precision = len(priors.required & true_arrows) / max(required_length, 1)
    required_recall = len(priors.required & true_arrows) / len(true_arrows)
    required_f1 = (
        2
        * required_precision
        * required_recall
        / max(required_precision + required_recall, 1e-6)
    )

    return {
        "priors": priors.model_dump(mode="json"),
        "forbidden_length": forbidden_length,
        "forbidden_Precision": forbidden_precision,
        "forbidden_Recall": forbidden_recall,
        "forbidden_F1": forbidden_f1,
        "required_length": required_length,
        "required_Precision": required_precision,
        "required_Recall": required_recall,
        "required_F1": required_f1,
    }


async def evaluate_priors(
    bif_paths: list[str | Path],
    prior_model: str,
    parse_model: str = None,
    exclude_descriptions: bool = False,
) -> pd.DataFrame:
    if parse_model is None:
        parse_model = prior_model
    prior_results = []
    prior_tasks = []
    for bif_path in bif_paths:
        if isinstance(bif_path, str):
            bif_path = Path(bif_path)
        bn = gum.loadBN(str(bif_path))
        variable_descriptions = {
            name: bn.variable(name).description() for name in bn.names()
        }
        if exclude_descriptions or not any(variable_descriptions.values()):
            variable_descriptions = None
        prior_results.append(
            {
                "bn": bn,
                "title": bn.propertyWithDefault("name", "no_name"),
                "filename": bif_path.stem,
                "num_nodes": bn.size(),
                "num_edges": len(bn.arcs()),
                "variable_descriptions": variable_descriptions,
            }
        )
        prior_tasks.append(
            generate_priors(
                bn=bn,
                variable_descriptions=variable_descriptions,
                prior_model=prior_model,
                parse_model=parse_model,
            )
        )
    prior_res = await tqdm.gather(*prior_tasks)
    for res_dict, prior_res in zip(prior_results, prior_res):
        res_dict.update(prior_res)

    return pd.DataFrame(prior_results)

### Define experiment and helper function to evaluate the precision of LLM removed edges as causal priors

In [ ]:
async def run_experiments(
    dataset_paths,
    n_runs=5,
    exclude_descriptions=True,
    prior_model="gemini-2.5-flash",
    parse_model=None,
):
    """
    Run multiple experiments and return combined results.

    Usage:
        results_no_desc = await run_experiments(bnlearn_small_datasets, n_runs=5, exclude_descriptions=True)
        results_with_desc = await run_experiments(bnlearn_small_datasets, n_runs=5, exclude_descriptions=False)
    """
    all_results = []

    for run_id in range(n_runs):
        print(f"Run {run_id + 1}/{n_runs}")
        df = await evaluate_priors(
            bif_paths=dataset_paths,
            prior_model=prior_model,
            parse_model=parse_model,
            exclude_descriptions=exclude_descriptions,
        )
        df["run_id"] = run_id
        all_results.append(df)

    return pd.concat(all_results, ignore_index=True)


def get_summary(df, metrics=["Precision", "Recall", "F-beta"]):
    """
    Get summary statistics for each dataset.

    Usage:
        summary = get_summary(results_no_desc)
    """
    summary_data = []

    for dataset in df["filename"].unique():
        dataset_df = df[df["filename"] == dataset]

        for metric in metrics:
            values = dataset_df[metric].dropna()

            if len(values) > 0:
                summary_data.append(
                    {
                        "Dataset": dataset,
                        "Metric": metric,
                        "Mean": values.mean(),
                        "Std": values.std(),
                        "Min": values.min(),
                        "Max": values.max(),
                        "Runs": len(values),
                    }
                )

    return pd.DataFrame(summary_data)


def show_report(df, metrics=["Precision", "Recall", "F-beta"]):
    """
    Get summary statistics for each dataset.

    Usage:
        summary = show_report(results_no_desc)
    """
    groups = df.groupby(["filename", "with_desc"])
    meta = groups.agg(
        {
            "num_nodes": "first",
            "num_edges": "first",
        }
    )
    meta["repeats"] = groups.size()
    meta.columns = pd.MultiIndex.from_product([["meta"], meta.columns])
    index = (
        groups["num_nodes"]
        .first()
        .reset_index()
        .sort_values(["num_nodes", "filename", "with_desc"])
        .set_index(["filename", "with_desc"])
        .index
    )
    summary = pd.concat([meta, groups[metrics].agg(["mean", "std"])], axis=1)
    return summary.loc[index]

### Experiment Configurations

In [ ]:
repeats = 5
prior_model = "gemini-2.5-flash"
parse_model = "gemini-2.5-flash-lite"


In [ ]:
synthetic_datasets = list(Path("synthetic").glob("*.bifxml"))

In [ ]:
synthetic_results_with_desc = await run_experiments(
    synthetic_datasets,
    n_runs=repeats,
    prior_model=prior_model,
    parse_model=parse_model,
    exclude_descriptions=False,
)
synthetic_results_with_desc.drop(columns=["bn"]).to_json("results/llm_constraints/synthetic-desc.json", orient="records", indent=4)
synthetic_results_with_desc

In [ ]:
synthetic_results_without_desc = await run_experiments(
    synthetic_datasets,
    n_runs=repeats,
    prior_model=prior_model,
    parse_model=parse_model,
    exclude_descriptions=True,
)
synthetic_results_without_desc.drop(columns=["bn"]).to_json("results/llm_constraints/synthetic.json", orient="records", indent=4)
synthetic_results_without_desc

### Run experiments on bnlearn datasets with/without variable descriptions

In [ ]:
bnlearn_small_datasets = list(Path("bnlearn/").glob("*.bifxml"))

In [ ]:
bnlearn_results_with_desc = await run_experiments(
    bnlearn_small_datasets,
    n_runs=repeats,
    exclude_descriptions=False,
    prior_model=prior_model,
    parse_model=parse_model,
)
# Save complete results for the record
bnlearn_results_with_desc.drop(columns="bn").to_json("results/llm_constraints/bnlearn-desc.json", orient="records", indent=4)

In [ ]:
bnlearn_results_without_desc = await run_experiments(
    bnlearn_small_datasets,
    n_runs=repeats,
    exclude_descriptions=True,
    prior_model=prior_model,
    parse_model=parse_model,
)
# Save complete results for the record
bnlearn_results_without_desc.drop(columns="bn").to_json("results/llm_constraints/bnlearn.json", orient="records", indent=4)

## Priors Aggregation

We use intersection of all runs to get the consensus

In [ ]:
from pathlib import Path

import pyagrum as gum
import pandas as pd

from priors.schema import Constraints


def aggregate_priors(priors_path: str, bifs_path: str):
    res = []
    df = pd.read_json(priors_path)
    for dataset in df["filename"].unique():
        bn = gum.loadBN(str(Path(bifs_path) / f"{dataset}.bifxml"))
        true_arrows = {
            (bn.variable(id1).name(), bn.variable(id2).name()) for id1, id2 in bn.arcs()
        }

        df_sub = df[df["filename"] == dataset]
        priors = [Constraints(**prior_dict) for prior_dict in df_sub["priors"]]
        majority_priors = Constraints(
            forbidden=set.intersection(*[priors.forbidden for priors in priors]),
            required=set.intersection(*[priors.required for priors in priors]),
        )

        forbidden_metrics = {
            "forbidden_length": len(majority_priors.forbidden),
            "forbidden_Precision": len(majority_priors.forbidden - true_arrows)
            / max(len(majority_priors.forbidden), 1),
            "forbidden_Recall": len(majority_priors.forbidden - true_arrows)
            / (bn.size() * (bn.size() - 1) - len(true_arrows)),
        }
        forbidden_metrics["forbidden_F1"] = (
            2
            * forbidden_metrics["forbidden_Precision"]
            * forbidden_metrics["forbidden_Recall"]
            / max(
                forbidden_metrics["forbidden_Precision"]
                + forbidden_metrics["forbidden_Recall"],
                1e-6,
            )
        )

        required_metics = {
            "required_length": len(majority_priors.required),
            "required_Precision": len(majority_priors.required & true_arrows)
            / max(len(majority_priors.required), 1),
            "required_Recall": len(majority_priors.required & true_arrows)
            / (len(true_arrows)),
        }
        required_metics["required_F1"] = (
            2
            * required_metics["required_Precision"]
            * required_metics["required_Recall"]
            / max(
                required_metics["required_Precision"]
                + required_metics["required_Recall"],
                1e-6,
            )
        )
        res.append(
            {
                **df_sub[
                    ["filename", "num_nodes", "num_edges", "variable_descriptions"]
                ]
                .iloc[0]
                .to_dict(),
                "priors": majority_priors.model_dump(mode="json"),
                **forbidden_metrics,
                **required_metics,
            }
        )

    return pd.DataFrame(res)

In [ ]:
bnlearn_desc_priors_aggregated = aggregate_priors(
    "results/llm_constraints/bnlearn-desc.json", "bnlearn"
)
bnlearn_desc_priors_aggregated.to_json("results/llm_constraints/bnlearn-desc-consensus.json", orient="records", indent=4)
bnlearn_desc_priors_aggregated

In [ ]:
bnlearn_priors_aggregated = aggregate_priors(
    "results/llm_constraints/bnlearn.json", "bnlearn"
)
bnlearn_priors_aggregated.to_json("results/llm_constraints/bnlearn-consensus.json", orient="records", indent=4)
bnlearn_priors_aggregated

In [ ]:
synthetic_desc_priors_aggregated = aggregate_priors(
    "results/llm_constraints/synthetic-desc.json", "synthetic"
)
synthetic_desc_priors_aggregated.to_json("results/llm_constraints/synthetic-desc-consensus.json", orient="records", indent=4)
synthetic_desc_priors_aggregated

In [ ]:
synthetic_priors_aggregated = aggregate_priors(
    "results/llm_constraints/synthetic.json", "synthetic"
)
synthetic_priors_aggregated.to_json("results/llm_constraints/synthetic-consensus.json", orient="records", indent=4)
synthetic_priors_aggregated